# Predicción de Precios de Viviendas en California

**Objetivo**  
En esta actividad, analizaremos datos de un censo del estado de California para construir un modelo de aprendizaje automático que prediga el precio medio de las viviendas en diferentes distritos del estado.  

**Descripción del Conjunto de Datos**  
El conjunto de datos incluye información sobre diversos factores que pueden influir en el precio de las viviendas, como:  
- **Población:** Número de habitantes en cada área.  
- **Ingreso medio:** Promedio de ingresos de los residentes en cada distrito.  
- **Precio medio de las viviendas:** Valor promedio de las propiedades en cada zona.  

Los datos están organizados por **grupos de bloques**, que constituyen la unidad geográfica más pequeña utilizada por la Oficina del Censo de los Estados Unidos para la publicación de información estadística. Cada grupo de bloques generalmente tiene una población de entre **600 y 3,000 personas**, nos referiremos a estos grupos como **"distritos"**.  

### **Data pipeline o flujo de datos**

Una secuencia de componentes de procesamiento de datos se llama flujo de datos, _data pipeline_ o simplemente _pipeline_. Los _pipelines_ son muy comunes en sistemas de aprendizaje automático, ya que hay muchos datos para manipular y muchas transformaciones de datos que aplicar.

Nosotros vamos a utilizar este concepto de _pipeline_ para transformar los datos, aplicar algún algoritmo (modelo) y evaluar posteriormente dicho algoritmo. Particularmente utilizaremos herramientas de la biblioteca de scikit-learn junto con pandas.

**¿Qué es un ML Pipeline?**  

Un **pipeline de aprendizaje automático (ML Pipeline o Pipeline)** es una secuencia estructurada de pasos que se utilizan para preparar los datos, entrenar modelos y evaluar su desempeño de manera eficiente y reproducible. Su objetivo es automatizar el flujo de trabajo del aprendizaje automático, asegurando que cada fase del proceso se realice de manera consistente y sin errores manuales.


**Etapas de un Pipeline**  

1. **Adquisición y Generación de Datos**  
   - Obtener datos de distintas fuentes (archivos, bases de datos, APIs, sensores, etc.).  
   - Generar datos sintéticos si es necesario para aumentar el conjunto de entrenamiento.  

2. **Construcción del Dataset**  
   - Seleccionar las variables relevantes.  
   - Integrar diferentes fuentes de datos.  
   - Manejar datos faltantes o inconsistentes.  

3. **Análisis Exploratorio de Datos (EDA)**  
   - Examinar la distribución de las variables.  
   - Detectar valores atípicos y correlaciones.  
   - Visualizar patrones y tendencias.  

4. **División del Conjunto de Datos**  
   - Separar los datos en conjunto de entrenamiento, validación y prueba.  
   - Asegurar una distribución adecuada de las clases en cada conjunto.  

5. **Preprocesamiento de Datos**  
   - Manejo de datos faltantes.  
   - Normalización, estandarización de variables.  
   - Codificación de variables categóricas.  

6. **Selección y Entrenamiento del Modelo**  
   - Elegir el modelo adecuado según la tarea (regresión, clasificación, clustering, etc.).  
   - Ajustar los hiperparámetros para optimizar el rendimiento del modelo.  
   - Entrenar el modelo con los datos procesados.  

7. **Evaluación del Modelo**  
   - Medir el desempeño con métricas como MSE (Error Cuadrático Medio) para regresión o Precisión, Recall y F1-score para clasificación.  
   - Comparar diferentes modelos y configuraciones.  

8. **Despliegue del Modelo**  
   - Integrar el modelo en una aplicación o servicio en producción.  
   - Monitorizar su rendimiento en tiempo real y actualizarlo según sea necesario.  



**¿Por qué usar un Pipeline?**  

✅**Automatización:** Reduce la intervención manual en cada paso del proceso.  
✅**Reproducibilidad:** Permite replicar el flujo de trabajo con facilidad.  
✅**Escalabilidad:** Puede adaptarse a grandes volúmenes de datos sin modificar la estructura.  
✅**Modularidad:** Facilita la experimentación y prueba de diferentes enfoques sin afectar todo el sistema.  



# Análisis Exploratorio de Datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Cargamos los datos

In [2]:
df_housing = pd.read_csv("./1_datos/housing.csv")
df_housing.head()

FileNotFoundError: [Errno 2] No such file or directory: './1_datos/housing.csv'

Cada fila representa un distrito. La tabla tiene 10 atributos:
- longitud,
- latitud,
- mediana de la edad de viviendas,
- total de habitaciones,
- total de dormitorios,
- población,
- hogares,
- mediana del ingreso,
- mediana del valor de la vivienda y
- proximidad al océano.

**Para conocer los atributos existentes y el número de instancias pertenecientes a cada atributo**

In [ ]:
df_housing.info()

El conjunto de datos contiene **20,640 instancias**, pero el atributo `total_bedrooms` solo tiene **20,433** valores no nulos, lo que indica que hay 207 valores faltantes en esta columna. Podemos observar también información sobre los **tipos de datos**

**Para conocer las categorias existentes en un atributo categórico y el número de instancias pertenecientes a cada categoría**

In [ ]:
df_housing["ocean_proximity"].value_counts()

**describe() muestra un resumen de las estadísticas de los atributos numéricos**

In [ ]:
df_housing.describe()

In [ ]:
df_housing.hist(bins=50, edgecolor='k', figsize=(20,15))
plt.show()

- El atributo **`median_income`** no está expresado en dólares estadounidenses (USD). Tras verificarlo, se ha confirmado que los valores han sido **escalados y limitados**, con un mínimo de **0.5** y un máximo de **15**. Estos números representan aproximadamente **decenas de miles de dólares** (por ejemplo, un valor de `3` equivale a un ingreso de aproximadamente **$30,000**). Es común trabajar con atributos preprocesados en aprendizaje automático, lo que no necesariamente representa un problema, pero es fundamental comprender cómo se calcularon los datos para interpretarlos correctamente.  

- Los atributos **`housing_median_age`** y **`median_house_value`** también han sido **limitados**. En particular, **`median_house_value`** es el atributo objetivo (_target_), lo que podría generar un problema significativo: el modelo podría aprender que los precios nunca superan dicho límite (sesgo de medición). Si nuestro cliente necesita predicciones precisas incluso **más allá de $500,000**, existen dos opciones:  
  1. **Recopilar etiquetas adecuadas** para los distritos cuyos valores han sido limitados.  
  2. **Eliminar esos distritos del conjunto de datos**.  

- Algunos **histogramas presentan una distribución sesgada hacia la derecha**, es decir, se extienden mucho más hacia la derecha de la mediana que hacia la izquierda. Esta asimetría puede dificultar la detección de patrones para ciertos algoritmos de aprendizaje automático, lo que podría requerir transformaciones en los datos, como el uso de una escala logarítmica o normalización, para mejorar el rendimiento del modelo.

## Advertencia ⚠️

> Antes de continuar observando los datos, vamos a separar en un **conjunto de entrenamiento (train)** y en un **conjunto de prueba (test)**, a este último no lo miramos más!!!

¿Qué podemos hacer antes de dividir los datos?
Entender el formato, aplicar alguna regla fija (por ejemplo, convertir a minúsculas un string, dividir por 1000 porque sé que está en metros, sumar dos columnas para crear una feature, sin mirar estadísticas). El problema son las transformaciones que estiman algo del dataset.

**¿Qué es un muestreo estratificado?**

En muchos problemas de aprendizaje automático, es crucial que los conjuntos de entrenamiento y de prueba sean representativos de la distribución de los datos originales. Si esto no es así, las evaluaciones del modelo pueden ser engañosas.

Por ejemplo, si consultamos a expertos, podríamos descubrir que **median_income** es un atributo clave para predecir el precio medio de las viviendas. En este caso, debemos asegurarnos de que la distribución de ingresos de los conjuntos sean similares a la del conjunto de datos original. Para lograrlo, utilizamos una técnica llamada muestreo estratificado.

El muestreo aleatorio simple no siempre garantiza que todas las categorías importantes de una variable estén bien representadas en los conjuntos de entrenamiento y prueba. Para solucionar esto, el muestreo estratificado divide los datos en estratos basados en un atributo relevante y luego selecciona muestras proporcionales de cada estrato.

En este caso, median_income es un atributo numérico continuo, por lo que primero necesitamos convertirlo en un atributo categórico. Para ello, creamos una nueva variable llamada `income_cat`, donde agruparemos los valores de median_income en categorías o estratos.

**Análisis del histograma de median_income**

In [ ]:
_ = df_housing["median_income"].hist(bins=50, color='skyblue', edgecolor='k', figsize=(7,5))

Al observar el histograma de `median_income`, notamos que:  
- La mayoría de los valores están **concentrados entre 1.5 y 6**, lo que equivale a ingresos aproximados de **$15,000 a $60,000**.  
- Algunos valores superan **6**, lo que representa ingresos más altos y menos frecuentes.  
- Para que la muestra sea representativa, es importante que cada estrato tenga **suficientes instancias**; de lo contrario, los resultados podrían estar **sesgados** (sesgo en la muestra).  
- **No conviene definir demasiados estratos**, ya que algunos podrían contener muy pocos datos y no aportarían información útil.  


### **Creación de los estratos con `pd.cut()`**  
Para segmentar los ingresos en cinco categorías, utilizamos la función `pd.cut()`, que divide el rango de valores en intervalos definidos:  

- **Categoría 1:** `income_cat = 1` → ingresos entre **0 y 1.5** (menos de **$15,000**).  
- **Categoría 2:** `income_cat = 2` → ingresos entre **1.5 y 3**.  
- **Categoría 3:** `income_cat = 3` → ingresos entre **3 y 4.5**.  
- **Categoría 4:** `income_cat = 4` → ingresos entre **4.5 y 6**.  
- **Categoría 5:** `income_cat = 5` → ingresos **mayores a 6**.  

Este enfoque nos permite aplicar **muestreo estratificado** asegurando que la distribución de ingresos en el conjunto de prueba sea representativa de la población total, lo que mejora la capacidad del modelo para generalizar sus predicciones. El modelo puede parecer mejor o peor simplemente porque el conjunto de test no es representativo.

In [ ]:
df_housing["income_cat"] = pd.cut(df_housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])
df_housing.head()

In [ ]:
df_housing["income_cat"].value_counts().sort_index().plot.bar(rot=0)
plt.xlabel("Income category")
plt.ylabel("Number of districts")
_ = plt.grid()

Ahora podemos hacer un muestreo estratificado usando la categoría `income_cat`

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df_housing, test_size=0.2, stratify=df_housing["income_cat"], random_state=42)

In [ ]:
# Eliminamos la categoria income_cat de ambos conjuntos porque no la usamos más
for set_ in (train_set, test_set):
    set_.drop("income_cat", axis=1, inplace=True)

## Visualización del conjunto de entrenamiento

In [ ]:
# Antes de continuar hacemos una copia del conjunto de entrenamiento
housing = train_set.copy()

### Visualización geográfica

In [ ]:
housing.plot(kind = "scatter", x = "longitude", y = "latitude", alpha=0.2, figsize=(10,7))
plt.show()

In [ ]:
housing.plot(kind="scatter", x = "longitude", y = "latitude", alpha=0.4,
              s = housing["population"]/100 , label="población", c="median_house_value",
              cmap="jet", colorbar = True, legend = True, sharex = False, figsize=(10,8))
plt.show()

Esta imagen indica que los precios de las viviendas están muy relacionados con la ubicación (por ejemplo, cerca del océano) y con la densidad de población.

## Correlaciones en el conjunto de Datos

Para analizar las relaciones entre los atributos del conjunto de datos, podemos calcular fácilmente el **coeficiente de correlación de Pearson** entre cada par de variables. En particular, nos interesa conocer la correlación de cada atributo con la variable objetivo, `median_house_value`, ya que esto nos ayudará a identificar qué características tienen mayor impacto en el precio de las viviendas.

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix

### **¿Qué es el coeficiente de correlación de Pearson?**  

El **coeficiente de correlación de Pearson (r)** mide la fuerza y dirección de la relación **lineal** entre dos variables numéricas. Su valor oscila entre **-1 y 1**:  

- **r ≈ 1** → **Correlación positiva fuerte**: cuando una variable aumenta, la otra también tiende a aumentar.  
  - Ejemplo: Se observa una fuerte correlación positiva entre **`median_income`** y **`median_house_value`**, lo que indica que los ingresos medianos tienen un impacto significativo en el valor de las viviendas.  

- **r ≈ -1** → **Correlación negativa fuerte**: cuando una variable aumenta, la otra tiende a disminuir.  
  - Ejemplo: Existe una ligera correlación negativa entre **latitud** y **`median_house_value`**, lo que sugiere que las viviendas tienden a ser más económicas a medida que nos desplazamos hacia el norte.  

- **r ≈ 0** → **Sin correlación lineal**: las variables no tienen una relación lineal evidente. Sin embargo, esto no implica que no haya relación alguna, solo que no es lineal.

**Correlación de cada atributo con `median_house_value`**

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)


Al calcular las correlaciones con la variable objetivo, podemos obtener información valiosa sobre los factores que más influyen en los precios de las viviendas. En general, en este conjunto de datos:  

- **`median_income`** muestra la correlación más fuerte con **`median_house_value`** (cercana a **0.7** o más). Esto confirma que el nivel de ingresos en un distrito es un **indicador clave** para predecir el precio de las viviendas.  
- **`total_rooms` y `housing_median_age`** tienen correlaciones moderadas con el valor de las viviendas, lo que indica que pueden ser útiles, pero no son los principales factores determinantes.  
- **`latitude`** y **`longitude`** presentan correlaciones débiles, lo que sugiere que la ubicación geográfica tiene cierto impacto en los precios, pero de manera menos pronunciada en comparación con el ingreso medio.  

### **Limitaciones de la Correlación**  

Aunque la correlación de Pearson es una herramienta útil, **solo mide relaciones lineales**. Es posible que existan patrones no lineales entre los atributos que esta medida no detecte. Por ejemplo:  
- Una variable podría tener un **impacto significativo en los precios**, pero de forma no lineal (por ejemplo, los precios podrían aumentar hasta cierto punto y luego estabilizarse o disminuir).  
- La correlación no captura **efectos combinados** de múltiples atributos. Puede haber interacciones entre variables que afecten los precios de manera conjunta.

In [ ]:
from pandas.plotting import scatter_matrix

atributos = ["median_house_value", "median_income", "total_rooms", "housing_median_age", 'latitude']
scatter_matrix(housing[atributos], figsize=(15,10), hist_kwds={'bins':50, 'color':'salmon', 'edgecolor':'k'})
plt.show()

## Resumen de la primera parte: Análisis exploratorio de datos

En esta primera parte:
- Obtuvimos los datos
- Hicimos una exploración inicial para determinar:
    - número de instancias
    - número de atributos
    - tipos de datos de los atributos
    - si hay datos faltantes
- Dividimos los datos en conjuntos de entrenamiento y prueba. La división se hizo de forma estratificada en función de un atributo definido por expertos.
- Exploramos más en profundidad el conjunto de entrenamiento mediante visualizaciones, búsqueda de atributos correlacionados.


# Prepación de los Datos

### Separamos los predictores de las etiquetas

In [ ]:
housing_train = housing.drop("median_house_value", axis=1)
housing_labels = housing["median_house_value"].copy()

## Limpieza de Datos

La mayoría de los algoritmos de aprendizaje automático no pueden trabajar con características faltantes, por lo que debemos ocuparnos de esto. Por ejemplo, vimos que el atributo `total_bedrooms` tiene algunos valores faltantes. Existen tres opciones para solucionar esto:

1. Eliminar los distritos correspondientes.
2. Eliminar el atributo.
3. Reemplazar los valores faltantes con algún valor (cero, la media, la mediana, etc.). A esto se llama _imputación_.

In [ ]:
# índices de las filas con datos faltantes
null_rows_idx = housing_train.isnull().any(axis=1)
# housing.loc[null_rows_idx].head()
null_rows_idx

In [ ]:
housing_train.loc[null_rows_idx]

In [ ]:
housing_train.loc[null_rows_idx].info()

In [ ]:
housing_option1 = housing_train.copy()

housing_option1.dropna(subset=["total_bedrooms"], inplace=True)  # opción 1: elimino los distritos con datos faltantes

housing_option1.loc[null_rows_idx].head()

In [ ]:
housing_option2 = housing_train.copy()

housing_option2.drop("total_bedrooms", axis=1, inplace=True)  # opción 2: elimino la columna

housing_option2.loc[null_rows_idx].head()

In [ ]:
housing_option3 = housing_train.copy()

median = housing_option3["total_bedrooms"].median()
housing_option3["total_bedrooms"] = housing_option3["total_bedrooms"].fillna(median)  # opción 3: reemplazo los valores faltantes con la mediana

housing_option3.loc[null_rows_idx]

vamos a optar por la opción 3 ya que es la menos destructiva, pero en lugar del código anterior, usaremos una clase de Scikit-Learn:   `SimpleImputer`.

La ventaja que esto posee es que almacenará el valor de la mediana de cada característica. Esto permitirá imputar valores faltantes no solo en el conjunto de entrenamiento, sino también en el conjunto de pruebas y cualquier dato nuevo que ingrese al modelo.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

La mediana solo puede calcularse en atributos numéricos, creamos una copia de los datos solo con atributos numéricos (excluyendo `ocean_proximity`)

In [ ]:
housing_num = housing_train.drop("ocean_proximity", axis=1)
housing_num

In [ ]:
imputer.fit(housing_num)
imputer.statistics_


Ahora se puede usar este imputador "entrenado" para transformar el conjunto de entrenamiento reemplazando los valores faltantes con las medianas calculadas:

In [ ]:
X = imputer.transform(housing_num) # me devuelve un numpy array
X

Los valores faltantes también pueden reemplazarse con el valor medio `(strategy="mean")`, o con el valor más frecuente `(strategy="most_frequent")`, o con un valor constante `(strategy="constant", fill_value=...)`. Las dos últimas estrategias admiten datos no numéricos.

In [ ]:
imputer.feature_names_in_

In [ ]:
df_train = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
df_train.loc[null_rows_idx].head()

## Manejo de atributos categóricos

Nuestro conjunto de datos tiene el atributo `ocean_proximity` que es de tipo categórico. La mayoría de algoritmos de aprendizaje automático no trabajan con atributos categóricos. Es necesario transformarlos en un formato que los algoritmos de aprendizaje automático puedan interpretar. Sin embargo, la forma en que se codifican estas variables puede afectar significativamente el rendimiento del modelo.

In [ ]:
housing_cat = housing_train.loc[:,["ocean_proximity"]]
housing_cat.head(8)

### Codificación de Variables Categóricas: Ordinal Encoder vs One-Hot Encoder

Primero probaremos la clase `OrdinalEncoder` de Scikit-Learn

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

In [ ]:
housing_cat_encoded[:8]

In [ ]:
ordinal_encoder.categories_

**Problema con la Codificación Ordinal**
Una forma sencilla de convertir valores categóricos en números es mediante codificación ordinal (`OrdinalEncoder` en scikit-learn). Este método asigna un número único a cada categoría.

| `ocean_proximity`  | Codificación ordinal |
|--------------------|---------------------|
| `<1H OCEAN`       | 0                   |
| `INLAND`          | 1                   |
| `ISLAND`          | 2                   |
| `NEAR BAY`        | 3                   |
| `NEAR OCEAN`      | 4                   |

El problema con esta representación es que los algoritmos de aprendizaje automático asumirán que los valores más cercanos son más similares que los más distantes. Es decir, el modelo podría interpretar erróneamente que INLAND (1) está más relacionado con <1H OCEAN (0) que con NEAR OCEAN (4), lo cual no es cierto, ya que estas categorías no tienen un orden natural.

`OrdinalEncoder` es útil cuando las categorías tienen un orden lógico, como en una escala de evaluación:

`"malo" → 0`, `"regular" → 1`, `"bueno" → 2`, `"excelente" → 3.`

**One-Hot Encoding**  

Para evitar este problema, una alternativa más efectiva es la codificación One-Hot (`OneHotEncoder` en scikit-learn). En este método, se crean variables binarias (0 o 1) para cada categoría, de manera que solo una de ellas estará activa.  

| `ocean_proximity`  | `<1H OCEAN` | `INLAND` | `ISLAND` | `NEAR BAY` | `NEAR OCEAN` |
|--------------------|-------------|----------|----------|------------|--------------|
| `<1H OCEAN`       | 1           | 0        | 0        | 0          | 0            |
| `INLAND`          | 0           | 1        | 0        | 0          | 0            |
| `ISLAND`          | 0           | 0        | 1        | 0          | 0            |
| `NEAR BAY`        | 0           | 0        | 0        | 1          | 0            |
| `NEAR OCEAN`      | 0           | 0        | 0        | 0          | 1            |

Clase `OneHotEncoder` de Scikit-Learn

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder(handle_unknown="ignore")
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

Por defecto, la salida de un `OneHotEncoder` es una matriz rala (sparse matrix), en lugar de un arreglo NumPy. Una matriz rala es una representación muy eficiente para matrices que contienen principalmente ceros. Internamente, solo almacena los valores no nulos y sus posiciones. Cuando un atributo categórico tiene cientos o miles de categorías, codificarlo con one-hot resulta en una matriz muy grande llena de 0s excepto por un solo 1 por fila. Una matriz rala ahorra mucha memoria y acelerará los cálculos.

Se puede usar una matriz rala de manera similar a un arreglo 2D normal, pero se puede convertir a un arreglo NumPy (denso) con el método `toarray()`. También puede fijarse `sparse=False` al crear el `OneHotEncoder`, en cuyo caso el método `transform()` devolverá directamente un arreglo NumPy.

In [ ]:
housing_cat_1hot.toarray()

Cuando entrenamos cualquier estimador de Scikit-Learn utilizando un DataFrame, el estimador guarda los nombres de las columnas en el atributo `feature_names_in_`. Luego, Scikit-Learn asegura que cualquier DataFrame proporcionado a este estimador después de de ser entrenado (por ejemplo, a transform() o predict()) tenga los mismos nombres de columnas. También proporcionan un método `get_feature_names_out()` que se puede usar para construir un DataFrame:

In [ ]:
cat_encoder.feature_names_in_

In [ ]:
cat_encoder.get_feature_names_out()

In [ ]:
df_housing_cat_1hot = pd.DataFrame(housing_cat_1hot.toarray(),
                         columns=cat_encoder.get_feature_names_out(),
                         index=housing_cat.index)

df_housing_cat_1hot

## Transformación y escalado de características

El escalado de características (feature scaling) es una de las transformaciones más importantes en el preprocesamiento de datos. Con pocas excepciones, los algoritmos de aprendizaje automático no funcionan bien cuando los atributos numéricos de entrada tienen escalas muy diferentes.

En nuestro caso, los atributos presentan grandes diferencias en escala:

- `total_rooms` varía entre 6 y 39,320, lo que representa valores en un rango amplio.
- `median_income` varía entre 0 y 15, en un rango mucho más reducido.

Si no realizamos un escalado previo, atributos como `total_rooms` tendrán una influencia dominante en el modelo, lo que puede hacer que otras variables relevantes, como `median_income`, sean ignoradas.

Para abordar este problema, se pueden aplicar dos enfoques principales: escalado min-max (normalización) y estandarización.

### Escalado min-max

El **escalado min-max** (también llamado **normalización**) transforma los valores de un atributo para que se encuentren dentro de un rango específico, **por defecto entre 0 y 1**.  

Se calcula con la siguiente fórmula:  

$$
X' = \frac{X - X_{\min}}{X_{\max} - X_{\min}}
$$

donde:  
- $X$ es el valor original de la característica.  
- $X_{\min}$ es el valor mínimo en la columna.  
- $X_{\max}$ es el valor máximo en la columna.  
- $X'$ es el valor escalado.  

Scikit-Learn proporciona el transformer `MinMaxScaler`, que permite escalar los valores entre 0 y 1, o cualquier otro rango deseado mediante el hiperparámetro `feature_range`:

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

In [ ]:
housing_num_min_max_scaled

📌 Cuándo usarlo

- Recomendado para modelos basados en redes neuronales, ya que muchas arquitecturas funcionan mejor con valores en un rango pequeño (a menudo de -1 a 1).

- Útil cuando los datos tienen una distribución aproximadamente uniforme y no contienen valores atípicos extremos.

### Estandarización

La **estandarización** transforma los datos restando la media y dividiéndolos por la desviación estándar, de manera que la distribución resultante tenga **media 0** y **desviación estándar 1**.  

Se calcula con la siguiente fórmula:  

$$
X' = \frac{X - \mu}{\sigma}
$$

donde:  
- $X$ es el valor original.  
- $\mu$ es la media de la columna.  
- $\sigma$ es la desviación estándar de la columna.  
- $X'$ es el valor estandarizado.  

Scikit-Learn proporciona un _transformer_ llamado `StandardScaler`:

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)

In [ ]:
housing_num_std_scaled

📌 Cuándo usarla

- Si los datos tienen valores atípicos que podrían afectar el escalado min-max.

- Si el modelo utilizado asume datos centrados en cero, como SVM, regresión logística y PCA.

✅ Si se quiere escalar una matriz rala sin convertirla en una matriz densa, se puede utilizar `StandardScaler` con el hiperparámetro `with_mean` igual a `False`. Esto solo dividirá los datos por el desvío estándar, sin restar la media (sino dejaría de ser rala):

### Transformación de características

Cuando la distribución de una característica es muy sesgada, tanto el escalado min-max como la estandarización pueden no ser suficientes para mejorar el desempeño del modelo. Esto se debe a que ambos métodos tratan todos los valores de la misma manera, sin considerar la asimetría de la distribución.

En estos casos, antes de escalar un atributo, se recomienda transformarlo para reducir el sesgo y, si es posible, hacer que su distribución sea aproximadamente simétrica (normal o gaussiana). Esto puede mejorar la capacidad del modelo para capturar patrones en los datos.

Para características positivas con sesgo hacia la derecha, se puede aplicar una transformación de raíz cuadrada o elevarla a una potencia menor que 1 para comprimir los valores altos y expandir los valores pequeños.

Si la característica tiene una cola muy larga y pesada (es decir, valores muy grandes que afectan la distribución), se puede aplicar la transformación logarítmica

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 7), sharey=True)

housing_train["population"].hist(ax=axs[0], bins=50)
housing_train["population"].apply(np.log).hist(ax=axs[1], bins=50)

axs[0].set_xlabel("Population")
axs[1].set_xlabel("Log of population")
axs[0].set_ylabel("Number of districts")

plt.show()

Hasta ahora, solo hemos analizado las características de entrada, pero es posible que también sea necesario transformar la variable objetivo. Por ejemplo, si la distribución del target tiene un sesgo, podemos aplicarle el logaritmo. Sin embargo, el modelo predecirá el logaritmo del `median_house_value`, no el valor en sí. Deberíamos calcular la exponencial de la predicción para obtener el valor buscado.

## Resumen de la segunda parte: Preparación de los datos

En esta segunda parte:
- Hicimos una limpieza de los datos de entrenamiento:
    - Manejo de datos faltantes o erróneos. Imputamos valores (media, mediana, etc)
- Manejo de atributos categóricos
    - OrdinalEncoder
    - Codificación OneHot
- Transformación y escalado de atributos